In [7]:
import glob
import numpy as np
import polars as pl
import pandas as pd
from tqdm import tqdm

In [8]:
# Get gene trait associations
RAP_DIR = 'project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/REGENIE_results'

# ASSOC_FILE = 'loftee_mac20_associations_bh_corrected.parquet'
# ASSOC_FILE = 'regenie_127phenotypes_lofteeHC_mac20_EUR.parquet'
ASSOC_FILE = 'regenie_127phenotypes_lofteeHC_mac20_EUR_miss20per.parquet'

LOCAL_DIR = '/home/dnanexus/data_dir/'

!dx download {RAP_DIR}/{ASSOC_FILE} -o {LOCAL_DIR}

gene_trait_df = (
    pl.read_parquet(f'{LOCAL_DIR}/{ASSOC_FILE}')
    .filter(pl.col('pval_fdr')<=0.05)
    .select(['region', 'phenotype', 'pval_fdr'])
)

# CORR_FILE = 'regenie_127phenotypes_mac20_lofteeHC_EUR_correlations.parquet'
# !dx download {RAP_DIR}/{CORR_FILE} -o {LOCAL_DIR}

# loftee_corrs = (
#     pl.read_parquet(f'{LOCAL_DIR}/{CORR_FILE}')
#     .with_columns(
#         loftee_corr = pl.col('correlation'),
#         loftee_corr_abs = pl.col('correlation').abs(),
#         loftee_corr_dir = pl.col('correlation')/pl.col('correlation').abs(),
#     )
#     .select(['region', 'phenotype', 'loftee_corr', 'loftee_corr_abs', 'loftee_corr_dir']) 
# )

# gene_trait_df = (
#     gene_trait_df
#     .join(loftee_corrs, on=['region', 'phenotype'], how='inner')
#     .drop_nans()
#     .sort('loftee_corr_abs', descending=True)
#     .unique(subset=["region"], keep="first", maintain_order=True)
# )
gene_trait_df

Error: path "/home/dnanexus/data_dir/regenie_127phenotypes_lofteeHC_mac20_EUR_
miss20per.parquet" already exists but -f/--overwrite was not set


region,phenotype,pval_fdr
str,str,f64
"""ENSG00000132855""","""apolipoprotein_a_int""",0.000007
"""ENSG00000052841""","""apolipoprotein_a_int""",0.038502
"""ENSG00000110243""","""apolipoprotein_a_int""",0.003376
"""ENSG00000118137""","""apolipoprotein_a_int""",4.5099e-46
"""ENSG00000173064""","""apolipoprotein_a_int""",0.006545
…,…,…
"""ENSG00000182095""","""forced_expiratory_volume_in_1s…",0.02095
"""ENSG00000164741""","""forced_expiratory_volume_in_1s…",0.036208
"""ENSG00000205189""","""forced_expiratory_volume_in_1s…",0.045581


In [9]:
# EUR unrelated individuals

!dx download project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/sample_lists/unrelated_cauc_samples_3rd_degree.csv -o /home/dnanexus/data_dir/

unrel_eur_samples = pl.read_csv('/home/dnanexus/data_dir/unrelated_cauc_samples_3rd_degree.csv')['eid'].cast(pl.Utf8).to_list()
unrel_eur_samples[:5]

Error: path "/home/dnanexus/data_dir/unrelated_cauc_samples_3rd_degree.csv"
already exists but -f/--overwrite was not set


['1000020', '1000107', '1000161', '1000172', '1000221']

In [4]:
from scipy.special import ndtri

c = 3/8  # Blom's constant for inverse normal transformation (prevents infinite values at the tails)

# Download phenotypes: covariates and PRS corrected
!dx download project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/phenotypes/corrected_cov_PRS_traits_EUR.parquet -o /home/dnanexus/data_dir/

phenos = (
    pl.read_parquet('/home/dnanexus/data_dir/corrected_cov_PRS_traits_EUR.parquet')
    .rename({'individual':'sample'})
    .filter(pl.col('sample').is_in(unrel_eur_samples))
)

# phenos
long_phenos_int = (
    phenos
    .unpivot(
        index='sample',
        on=gene_trait_df['phenotype'].unique().to_list(),
        variable_name='phenotype',
        value_name='pheno_value'
    )
    .drop_nulls()

    .lazy()  # Use Lazy mode for better memory/query optimization
    .with_columns(
        # Calculate rank and group size using native Rust engine
        r = pl.col("pheno_value").rank().over("phenotype"),
        n = pl.len().over("phenotype")
    )
    .with_columns(
        # Calculate the INT value calling ndtri ONCE on the whole column
        pheno_value_int = ((pl.col("r") - c) / (pl.col("n") - 2*c + 1)).map_batches(ndtri)
    )
    .drop(["r", "n"]) # Clean up temporary columns
    .collect()
)

print(long_phenos_int['phenotype'].value_counts(sort=True))
long_phenos_int

Error: path "/home/dnanexus/data_dir/corrected_cov_PRS_traits_EUR.parquet"
already exists but -f/--overwrite was not set
shape: (102, 2)
┌─────────────────────────────────┬────────┐
│ phenotype                       ┆ count  │
│ ---                             ┆ ---    │
│ str                             ┆ u64    │
╞═════════════════════════════════╪════════╡
│ townsend_deprivation_index_at_… ┆ 378461 │
│ waist_circumference_int         ┆ 378281 │
│ hip_circumference_int           ┆ 378244 │
│ standing_height_int             ┆ 378108 │
│ weight_int                      ┆ 377841 │
│ …                               ┆ …      │
│ phosphate_int                   ┆ 330147 │
│ apolipoprotein_a_int            ┆ 328787 │
│ shbg_int                        ┆ 327605 │
│ testosterone_int                ┆ 327366 │
│ direct_bilirubin_int            ┆ 307355 │
└─────────────────────────────────┴────────┘


sample,phenotype,pheno_value,pheno_value_int
str,str,f64,f64
"""1000020""","""basophill_count_int""",2.897619,1.579456
"""1000107""","""basophill_count_int""",1.615834,0.491304
"""1000161""","""basophill_count_int""",1.150647,0.067196
"""1000172""","""basophill_count_int""",1.711554,0.579502
"""1000221""","""basophill_count_int""",1.04696,-0.022536
…,…,…,…
"""4974782""","""arm_fat_percentage_left_int""",0.291546,0.427302
"""5956310""","""arm_fat_percentage_left_int""",-1.022372,-1.486935
"""4301443""","""arm_fat_percentage_left_int""",-0.218325,-0.363237


In [5]:
long_phenos_int.filter(pl.col('phenotype')=='standing_height_int').select(pl.col('pheno_value').std())

pheno_value
f64
0.606581


In [6]:
long_phenos_int.filter(pl.col('phenotype')=='standing_height_int').select(pl.col('pheno_value_int').std())

pheno_value_int
f64
0.999992


In [10]:
mac = 20

RAP_ANNO_DIR = "project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/wgs/qced_maf1e-3_loftee_olink_genes/qced_maf1e-3_loftee_olink_genes_EURunrelated"

# ANNO_FILE = "annotations_fillna_ukbgym.parquet"
ANNO_FILE = "annotations_fillna_ukbgym_with_mane.parquet"

!dx download {RAP_ANNO_DIR}/{ANNO_FILE} -o {LOCAL_DIR}/{ANNO_FILE}

id_list = (
    pl.scan_parquet(f'{LOCAL_DIR}/{ANNO_FILE}')
    .filter(
        pl.col('region').is_in(gene_trait_df.select('region').unique().to_series()),
        pl.col('mac_ukb')<=mac,
    )
    .select('id')
    .unique()
    .collect()
)

id_list

Error: path
"/home/dnanexus/data_dir//annotations_fillna_ukbgym_with_mane.parquet" already
exists but -f/--overwrite was not set


/tmp/ipykernel_87358/3908164256.py:18: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  .collect()


id
str
"""chr2:71321215:A:C"""
"""chr7:107137982:G:T"""
"""chr19:2230227:C:T"""
"""chr22:40050745:C:T"""
"""chr19:11070613:G:T"""
…
"""chr3:79156579:G:A"""
"""chr19:13624144:GAA:G"""
"""chr7:82155548:T:G"""


In [8]:
# Download genotype (long gt) file
!dx download project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/wgs/qced_maf1e-3_loftee_olink_genes/qced_maf1e-3_loftee_olink_genes_EURunrelated/gt_long.parquet -o /home/dnanexus/data_dir/

long_gt = (
    pl.scan_parquet('/home/dnanexus/data_dir/gt_long.parquet')
    .select(['id', 'sample', 'gt'])
    .filter(
        pl.col('gt')==1,
        pl.col('sample').is_in(unrel_eur_samples),
    )
    .join(
        id_list.lazy(),
        on='id',
        how='semi'
    )

    .collect()
)

long_gt

Error: path "/home/dnanexus/data_dir/gt_long.parquet" already exists but
-f/--overwrite was not set


id,sample,gt
str,str,i8
"""chr10:24578613:TATAAA:T""","""1745607""",1
"""chr10:24578613:TATAAA:T""","""4369337""",1
"""chr10:24578613:TATAAA:T""","""3190490""",1
"""chr10:24578613:TATAAA:T""","""2067640""",1
"""chr10:24578613:TATAAA:T""","""1906931""",1
…,…,…
"""chr9:19312532:A:C""","""4707581""",1
"""chr9:19312532:A:C""","""5902273""",1
"""chr9:19312534:A:C""","""2004709""",1


In [14]:
import math

output_dir = '/home/dnanexus/data_dir/appv_phenos'
!mkdir -p {output_dir}

pheno_list = long_phenos_int['phenotype'].unique().to_list()
CHUNK_SIZE = 10
num_phenos = len(pheno_list)
num_chunks = math.ceil(num_phenos / CHUNK_SIZE)

# Process in Batches
for i in tqdm(range(0, num_phenos, CHUNK_SIZE)):
    # 1. Define the current batch of genes
    chunk_phenos = pheno_list[i : i + CHUNK_SIZE]

    print(f"Processing chunk starting at index: {i}")
    (
        long_phenos_int.lazy()
        .filter(pl.col('phenotype').is_in(chunk_phenos))
        .join(
            long_gt.lazy(),
            on='sample',
            how='inner'
        )
        .group_by(['id', 'phenotype'])
        .agg(
            n_individuals = pl.len().cast(pl.Int32),
            mean_pheno_value = pl.col('pheno_value_int').mean().cast(pl.Float32),
            std_pheno_value = pl.col('pheno_value_int').std().cast(pl.Float32),
        )
        
        # .with_columns(
        #     # Calculate rank and group size using native Rust engine
        #     r = pl.col("mean_pheno_value").rank().over("phenotype"),
        #     n = pl.len().over("phenotype")
        # )
        # .with_columns(
        #     # Calculate the INT value calling ndtri ONCE on the whole column
        #     mean_pheno_value_int = ((pl.col("r") - c) / (pl.col("n") - 2*c + 1)).map_batches(ndtri).cast(pl.Float32)
        # )
        # .drop(["r", "n"]) # Clean up temporary columns

        # .with_columns(
        #     mean_pheno_value_rank=pl.col('mean_pheno_value')
        #         .rank(method="max")
        #         .over('phenotype')
        #         .cast(pl.Float32),
        # )
        # .with_columns(
        #     mean_pheno_value_ptile=(
        #         pl.col('mean_pheno_value_rank') / pl.len().over('phenotype')
        #     ).cast(pl.Float32),
        # )

        .sink_parquet(f'{output_dir}/tmp_appv_chunk_{i}.parquet')
    )


  0%|          | 0/11 [00:00<?, ?it/s]

Processing chunk starting at index: 0


  9%|▉         | 1/11 [00:26<04:26, 26.61s/it]

Processing chunk starting at index: 10


 18%|█▊        | 2/11 [00:53<03:59, 26.63s/it]

...

 82%|████████▏ | 9/11 [03:49<00:50, 25.05s/it]

Processing chunk starting at index: 90


 91%|█████████ | 10/11 [04:15<00:25, 25.16s/it]

Processing chunk starting at index: 100


100%|██████████| 11/11 [04:20<00:00, 23.72s/it]


In [15]:
tmp = pl.read_parquet(f'{output_dir}/tmp_appv_chunk_0.parquet')
tmp

id,phenotype,n_individuals,mean_pheno_value,std_pheno_value
str,str,i32,f32,f32
"""chr10:24580217:T:C""","""platelet_distribution_width_in…",7,-0.102675,1.041814
"""chr10:24580563:A:G""","""direct_bilirubin_int""",10,-0.383603,1.175343
"""chr10:24583387:ATTACT:A""","""whole_body_fat_mass_int""",3,0.26512,0.050017
"""chr10:24586320:T:A""","""waist_circumference_int""",4,-0.719196,0.661429
"""chr10:24586602:CAT:C""","""whole_body_fat_mass_int""",12,-0.725646,1.030894
…,…,…,…,…
"""chr15:62768697:C:T""","""whole_body_fat_mass_int""",2,-0.01459,0.49463
"""chr15:62768802:C:T""","""apolipoprotein_b_int""",1,0.446637,null
"""chr15:62768859:A:C""","""standing_height_int""",3,0.343124,0.785425


In [17]:
(
    tmp
    .filter(pl.col('phenotype')=='apolipoprotein_b_int')
    .select(pl.col('mean_pheno_value').std())
)

mean_pheno_value
f32
0.799811


In [11]:
output_dir = '/home/dnanexus/data_dir/appv_phenos'
small_output_file = "/home/dnanexus/data_dir/quant_pheno_INT_loftee_mac20_EURunrelated_miss20per_appv_small.parquet"

# 1. Get list of files manually
files = glob.glob(f'{output_dir}/*.parquet')
print(f"Found {len(files)} files.")

# 2. Create a list of LazyFrames
lfs = [pl.scan_parquet(f) for f in files]

# 3. Concatenate with relaxation
appv_big = pl.concat(lfs, how="vertical_relaxed")


id_region = pl.scan_parquet(f'{LOCAL_DIR}/{ANNO_FILE}').select(['id', 'region']).unique()

(
    appv_big
    .join(id_region, on='id', how='inner')
    .join(gene_trait_df.lazy(), on=['region', 'phenotype'], how='inner')
    .drop(['region', 'pval_fdr'])
    .unique(subset=['id', 'phenotype'])          # ← deduplicate
    .sink_parquet(small_output_file, engine='streaming')
)

Found 11 files.


In [12]:
import polars as pl
small_output_file = "/home/dnanexus/data_dir/quant_pheno_INT_loftee_mac20_EURunrelated_miss20per_appv_small.parquet"
tmp = pl.read_parquet(small_output_file)
# tmp.head().collect()
tmp

id,phenotype,n_individuals,mean_pheno_value,std_pheno_value
str,str,i32,f32,f32
"""chr1:227114673:G:A""","""whole_body_fat_mass_int""",1,-0.025207,null
"""chr15:100087025:A:G""","""standing_height_int""",1,0.649444,null
"""chr9:114310398:G:A""","""trunk_fatfree_mass_int""",15,-0.294965,0.931101
"""chr8:94517468:G:C""","""trunk_predicted_mass_int""",1,1.601188,null
"""chr15:88830412:A:G""","""leg_fatfree_mass_left_int""",1,0.658579,null
…,…,…,…,…
"""chr15:64631201:G:C""","""hand_grip_strength_right_int""",3,-0.167021,0.568194
"""chr1:212107774:T:C""","""standing_height_int""",1,0.150419,null
"""chr9:110408556:G:A""","""mean_platelet_thrombocyte_volu…",1,-1.881508,null


In [13]:
# !dx upload {combined_output_file} --path project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/ukbgym/avg_pheno_per_var/

!dx upload {small_output_file} --path project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/ukbgym/avg_pheno_per_var/

[===========================================================>] Uploaded 872,915,124 of 872,915,124 bytes (100%) /home/dnanexus/data_dir/quant_pheno_INT_loftee_mac20_EURunrelated_miss20per_appv_small.parquet
ID                                file-J67gK90Jg0yPPVzF5kgy6Y37
Class                             file
Project                           project-Gyp4fvjJg0yFZ374KvP9bGFJ
Folder                            /processed_data/ukbgym/avg_pheno_per_var
Name                              quant_pheno_INT_loftee_mac20_EURunrelated_miss20per_appv_small.par
                                  quet
State                             closing
Visibility                        visible
Types                             -
Properties                        -
Tags                              -
Outgoing links                    -
Created                           Fri Feb 13 19:10:28 2026
Created by                        shubhankar
 via the job                      job-J67fBpQJg0yK5Q95fKq3Vxqz
Last modified